# ✂️ Rule-Based English Stemmer dari Nol

**Tugas Text Mining — Minggu 3**

Membangun *stemmer* bahasa Inggris berbasis aturan (*suffix stripping*) tanpa library NLP:
1. Versi sederhana — memotong sufiks `-ing`, `-ed`, `-s`, `-es`
2. Versi lanjutan — daftar sufiks lebih lengkap, panjang minimum stem, daftar pengecualian, dan perbaikan ejaan setelah pemotongan (mis. `studies → study`, `running → run`), lalu diuji terhadap hasil yang diharapkan.

## 1. Stemmer Sederhana

In [3]:
def simple_stemmer(word):
    suffixes = ["ing", "ed", "s", "es"]

    for suffix in suffixes:
        if word.endswith(suffix):
            return word[:-len(suffix)]

    return word

print(simple_stemmer("playing"))

words = ["playing", "played", "cars", "houses"]

for word in words:
    print(word, "->", simple_stemmer(word))

play
playing -> play
played -> play
cars -> car
houses -> house


## 2. Stemmer Lanjutan + Pengujian

In [2]:
import re
 
MIN_STEM_LENGTH = 3
EXCEPTIONS = {
    "this", "his", "its", "as", "was", "is", "has", "gas", "bus",
    "sing", "king", "ring", "thing", "bring", "string", "spring",
    "bed", "red", "wed", "need", "seed", "feed", "speed",
    "news", "series", "species", "always", "yes", "less",
}
 
SUFFIXES = ["ational", "ization", "fulness", "ousness", "iveness",
            "tion", "ment", "ness", "able", "ible", "ing",
            "ies", "ed", "ly", "es", "er", "est", "s"]
 
 
def _fix_spelling(stem, original):
    """Perbaiki ejaan setelah pemotongan sufiks."""
    if original.endswith("ies") and len(stem) >= 2:
        return stem + "y"
    if len(stem) >= 3 and stem[-1] == stem[-2] and stem[-1] not in "aeiou":
        if stem[-1] not in "lsz": 
            return stem[:-1]
    if original.endswith("es") and stem and stem[-1] not in "aeiou":
        if stem[-1] in "cgsvz" or stem.endswith(("th", "ch", "sh")) is False:
            return stem + "e"
    return stem
 
 
def simple_stemmer(word):
    word = word.lower().strip()
    word = re.sub(r"[^a-z]", "", word)
 
    if not word or word in EXCEPTIONS:
        return word
 
    for suffix in SUFFIXES:
        if word.endswith(suffix):
            stem = word[:-len(suffix)]
            if len(stem) >= MIN_STEM_LENGTH:
                return _fix_spelling(stem, word)
    return word
 
 
def stem_sentence(text):
    tokens = re.findall(r"[A-Za-z]+", text)
    return [simple_stemmer(t) for t in tokens]
 
 
def evaluate(test_cases):
    benar = 0
    print(f"{'kata':<14}{'hasil':<12}{'harapan':<12}status")
    print("-" * 46)
    for kata, harapan in test_cases.items():
        hasil = simple_stemmer(kata)
        cocok = hasil == harapan
        benar += cocok
        print(f"{kata:<14}{hasil:<12}{harapan:<12}{'OK' if cocok else 'SALAH'}")
    akurasi = benar / len(test_cases) * 100
    print("-" * 46)
    print(f"Akurasi: {benar}/{len(test_cases)} ({akurasi:.1f}%)")
    return akurasi
 
 
if __name__ == "__main__":
    TEST = {
        "playing": "play", "played": "play", "cars": "car",
        "houses": "house", "running": "run", "studies": "study",
        "sing": "sing", "king": "king", "bed": "bed", "bus": "bus",
        "this": "this", "was": "was", "happiness": "happy",
        "quickly": "quick", "movement": "move", "boxes": "box",
    }
    evaluate(TEST)
 
    print("\nContoh kalimat:")
    kalimat = "The cats were playing and running quickly!"
    print(" ", kalimat)
    print(" ", " ".join(stem_sentence(kalimat)))
 

kata          hasil       harapan     status
----------------------------------------------
playing       play        play        OK
played        play        play        OK
cars          car         car         OK
houses        house       house       OK
running       run         run         OK
studies       study       study       OK
sing          sing        sing        OK
king          king        king        OK
bed           bed         bed         OK
bus           bus         bus         OK
this          this        this        OK
was           was         was         OK
happiness     happi       happy       SALAH
quickly       quick       quick       OK
movement      move        move        OK
boxes         boxe        box         SALAH
----------------------------------------------
Akurasi: 14/16 (87.5%)

Contoh kalimat:
  The cats were playing and running quickly!
  the cat were play and run quick
